# Heart ECG Project Runner

Starter notebook for the MIT-BIH ECG project. The notebook provides dataset loading, device setup, window creation, and evaluation metrics. Students fill in preprocessing, model architecture, loss function, and optimizer.

## Initialize Project

- Import project dependencies
- Seed PyTorch for repeatable runs
- Detect the best available device: GPU, Apple Silicon, or CPU
- Use `wfdb` to read PhysioNet ECG records

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support

try:
    import wfdb
except ImportError as error:
    raise ImportError("Install dependencies with: pip install -r ../../requirements.txt") from error

torch.manual_seed(42)


def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


device = get_device()
device

## Locate Dataset

- Expect dataset files under `sandbox/heart/data`
- Run `python setup.py` from `sandbox/heart` if the dataset has not been downloaded
- Discover the standard MIT-BIH ECG records from top-level `.hea` header files

In [ ]:
CURRENT_DIR = Path.cwd()
HEART_DIR = CURRENT_DIR if CURRENT_DIR.name == "heart" else CURRENT_DIR / "sandbox" / "heart"
DATA_DIR = HEART_DIR / "data"
DATASET_DIR = DATA_DIR / "mit-bih-arrhythmia-database-1.0.0"

if not DATASET_DIR.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATASET_DIR}. Run `python setup.py` from sandbox/heart first."
    )

record_paths = sorted(DATASET_DIR.glob("*.hea"))
record_stems = [path.with_suffix("") for path in record_paths]

print(f"Found {len(record_stems)} ECG records")
record_stems[:5]

## Load ECG Records

- Read ECG signal arrays with `wfdb.rdrecord`
- Read beat annotations with `wfdb.rdann`
- Keep signal data, sampling rate, annotation locations, and annotation labels together
- Load all records by default so evaluation is less sensitive to a tiny sample

In [ ]:
def load_record(record_stem):
    record = wfdb.rdrecord(str(record_stem))
    annotation = wfdb.rdann(str(record_stem), "atr")
    return {
        "name": record_stem.name,
        "signals": record.p_signal.astype(np.float32),
        "fs": record.fs,
        "annotation_samples": np.asarray(annotation.sample),
        "annotation_symbols": np.asarray(annotation.symbol),
    }


def load_records(record_stems, limit=None):
    selected_records = record_stems[:limit] if limit else record_stems
    return [load_record(record_stem) for record_stem in selected_records]


raw_records = load_records(record_stems, limit=None)
raw_records[0].keys()

## Preprocess Records

- Fill in project-specific signal cleanup here
- Examples: select leads, normalize signals, remove noise, resample, or filter beats
- This function is intentionally empty but still gets called by the notebook
- Return the updated record dictionary

In [ ]:
def preprocess_record(record):
    # TODO: Student preprocessing goes here.
    # Examples:
    # record["signals"] = record["signals"][:, [0]]  # keep lead 0 only
    # record["signals"] = (record["signals"] - record["signals"].mean(axis=0)) / record["signals"].std(axis=0)
    return record


records = [preprocess_record(record) for record in raw_records]
records[0]["signals"].shape

## Create Model Inputs

- Convert long ECG signals into fixed-size windows
- Create a starter binary label for each window
- Label `0`: window contains only normal beat annotations
- Label `1`: window contains at least one non-normal beat annotation
- Students can replace this labeling rule with a better project-specific target

In [ ]:
NORMAL_SYMBOLS = {"N"}


def label_window(annotation_samples, annotation_symbols, start, end):
    in_window = (annotation_samples >= start) & (annotation_samples < end)
    symbols = annotation_symbols[in_window]
    if len(symbols) == 0:
        return None
    return int(any(symbol not in NORMAL_SYMBOLS for symbol in symbols))


def create_windows(records, window_size=360, step_size=180, lead_index=0):
    windows = []
    labels = []

    for record in records:
        signal = record["signals"][:, lead_index]
        samples = record["annotation_samples"]
        symbols = record["annotation_symbols"]

        for start in range(0, len(signal) - window_size, step_size):
            end = start + window_size
            label = label_window(samples, symbols, start, end)
            if label is None:
                continue
            windows.append(signal[start:end])
            labels.append(label)

    X = torch.tensor(np.asarray(windows), dtype=torch.float32).unsqueeze(1)
    y = torch.tensor(labels, dtype=torch.long)
    return X, y


X, y = create_windows(records)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("class counts:", torch.bincount(y))

## Visualize Dataset

- Plot class balance for normal vs abnormal windows
- Display one ECG window so students can inspect the model input
- Use these visuals to sanity-check preprocessing before training

In [ ]:
class_names = ["normal", "abnormal"]
class_counts = torch.bincount(y, minlength=2).cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar(class_names, class_counts)
axes[0].set_title("Window class balance")
axes[0].set_ylabel("count")

sample_index = 0
axes[1].plot(X[sample_index, 0].cpu().numpy())
axes[1].set_title(f"Sample ECG window: {class_names[y[sample_index].item()]}")
axes[1].set_xlabel("sample")
axes[1].set_ylabel("signal value")

plt.tight_layout()

## Build DataLoaders

- Use the standard inter-patient MIT-BIH DS1/DS2 record split
- Keep entire ECG records out of training so test metrics measure unseen-record generalization
- Create `DataLoader`s so training can use mini-batches

In [ ]:
DS1_TRAIN_RECORDS = [
    "101", "106", "108", "109", "112", "114", "115", "116", "118", "119", "122",
    "124", "201", "203", "205", "207", "208", "209", "215", "220", "223", "230",
]
DS2_TEST_RECORDS = [
    "100", "103", "105", "111", "113", "117", "121", "123", "200", "202", "210",
    "212", "213", "214", "219", "221", "222", "228", "231", "232", "233", "234",
]


def split_records(records):
    records_by_name = {record["name"]: record for record in records}
    missing_records = sorted(
        set(DS1_TRAIN_RECORDS + DS2_TEST_RECORDS) - set(records_by_name)
    )
    if missing_records:
        raise ValueError(f"Missing expected MIT-BIH records: {missing_records}")

    train_records = [records_by_name[name] for name in DS1_TRAIN_RECORDS]
    test_records = [records_by_name[name] for name in DS2_TEST_RECORDS]
    excluded_records = sorted(set(records_by_name) - set(DS1_TRAIN_RECORDS) - set(DS2_TEST_RECORDS))
    return train_records, test_records, excluded_records


train_records, test_records, excluded_records = split_records(records)
X_train, y_train = create_windows(train_records)
X_test, y_test = create_windows(test_records)

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128)

print("train records:", [record["name"] for record in train_records])
print("test records:", [record["name"] for record in test_records])
print("excluded records:", excluded_records)
print("train window class counts:", torch.bincount(y_train, minlength=2))
print("test window class counts:", torch.bincount(y_test, minlength=2))

len(train_dataset), len(test_dataset)

## Define Model

- Students define the model here
- Input shape is `(batch, channels, samples)`
- Current starter windows have `1` channel and `360` time samples
- Replace the TODO sections with a feedforward, 1D CNN, RNN, or another architecture

In [ ]:
class HeartECGModel(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: Define model layers here.
        # Example layer types to consider:
        # self.conv = nn.Conv1d(...)
        # self.rnn = nn.GRU(...)
        # self.classifier = nn.Linear(...)
        raise NotImplementedError("Define your model layers in __init__")

    def forward(self, x):
        # TODO: Define the forward pass.
        raise NotImplementedError("Define your model forward pass")


model = HeartECGModel().to(device)
model

## Choose Loss and Optimizer

- Students choose a loss function for the project target
- Students choose an optimizer and learning rate
- For the starter binary labels, `nn.CrossEntropyLoss()` is a reasonable first choice
- Adam is a common first optimizer, but students should be able to experiment

In [ ]:
# TODO: Choose a loss function.
# Example: loss_fn = nn.CrossEntropyLoss()
loss_fn = None

# TODO: Choose an optimizer.
# Example: optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
optimizer = None

if loss_fn is None:
    raise NotImplementedError("Choose a loss function before training")
if optimizer is None:
    raise NotImplementedError("Choose an optimizer before training")

## Train Model

- Train the model on ECG windows
- Move each batch to the selected device
- Run forward pass, loss calculation, backpropagation, and optimizer update
- Track loss and accuracy each epoch

In [ ]:
def evaluate_loss_accuracy(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_X, batch_y in loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            logits = model(batch_X)
            loss = loss_fn(logits, batch_y)
            total_loss += loss.item()
            correct += (logits.argmax(dim=1) == batch_y).sum().item()
            total += batch_y.size(0)

    return total_loss / len(loader), correct / total


history = {"train_loss": [], "train_accuracy": [], "test_loss": [], "test_accuracy": []}

for epoch in range(5):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        logits = model(batch_X)
        loss = loss_fn(logits, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (logits.argmax(dim=1) == batch_y).sum().item()
        total += batch_y.size(0)

    train_loss = total_loss / len(train_loader)
    train_accuracy = correct / total
    test_loss, test_accuracy = evaluate_loss_accuracy(model, test_loader, loss_fn)

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["test_loss"].append(test_loss)
    history["test_accuracy"].append(test_accuracy)

    print(
        f"epoch {epoch + 1}: "
        f"train_loss={train_loss:.4f}, train_accuracy={train_accuracy:.3f}, "
        f"test_loss={test_loss:.4f}, test_accuracy={test_accuracy:.3f}"
    )

## Plot Training Curves

- Plot train/test loss across epochs
- Plot train/test accuracy across epochs
- Use the curves to judge whether the model is learning, overfitting, or underfitting

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(epochs, history["train_loss"], label="train loss")
axes[0].plot(epochs, history["test_loss"], label="test loss")
axes[0].set_title("Loss")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("cross entropy")
axes[0].legend()

axes[1].plot(epochs, history["train_accuracy"], label="train accuracy")
axes[1].plot(epochs, history["test_accuracy"], label="test accuracy")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("accuracy")
axes[1].set_ylim(0, 1)
axes[1].legend()

plt.tight_layout()

## Evaluate Model

- Run the trained model on held-out ECG windows
- Report accuracy, precision, recall, F1-score, confusion matrix, and class-level metrics
- Plot the confusion matrix for quick visual inspection
- Use these metrics to judge performance beyond raw accuracy
- Precision and recall are especially useful when classes are imbalanced

In [ ]:
def collect_predictions(model, loader):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch_X, batch_y in loader:
            batch_X = batch_X.to(device)
            logits = model(batch_X)
            predictions = logits.argmax(dim=1).cpu()
            y_true.extend(batch_y.tolist())
            y_pred.extend(predictions.tolist())

    return np.asarray(y_true), np.asarray(y_pred)


y_true, y_pred = collect_predictions(model, test_loader)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="binary",
    zero_division=0,
)
cm = confusion_matrix(y_true, y_pred)

print(f"accuracy:          {accuracy_score(y_true, y_pred):.3f}")
print(f"balanced accuracy: {balanced_accuracy_score(y_true, y_pred):.3f}")
print(f"precision:         {precision:.3f}")
print(f"recall:            {recall:.3f}")
print(f"f1-score:          {f1:.3f}")
print("\nconfusion matrix:")
print(cm)
print("\nclassification report:")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(class_names)), class_names)
ax.set_yticks(range(len(class_names)), class_names)
ax.set_xlabel("predicted")
ax.set_ylabel("true")
ax.set_title("Confusion Matrix")

for row in range(cm.shape[0]):
    for col in range(cm.shape[1]):
        ax.text(col, row, cm[row, col], ha="center", va="center")

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()